In [0]:
from dataclasses import dataclass
from pyspark.sql import DataFrame

@dataclass
class Config:
    bronze_path_cliente: str =  (
        "abfss://raw@stbancoorion01.dfs.core.windows.net/"
        "bronze/clientes/"
    )

    bronze_path_contas: str = (
        "abfss://raw@stbancoorion01.dfs.core.windows.net/bronze/contas/"
    )

    bronze_path_emprestimos: str = (
        "abfss://raw@stbancoorion01.dfs.core.windows.net/bronze/emprestimos/"
    )

    bronze_path_pagamentos: str = (
        "abfss://raw@stbancoorion01.dfs.core.windows.net/bronze/pagamentos/"
    )

    bronze_path_transferencias: str = (
        "abfss://raw@stbancoorion01.dfs.core.windows.net/bronze/transferencias/"
    )

def salvar_delta(df_cliente: DataFrame, df_contas: DataFrame, df_emprestimos: DataFrame, df_pagamentos: DataFrame, df_transferencias: DataFrame):
    try:
        df_cliente.write.format("delta").mode("append") \
        .option("path", Config.bronze_path_cliente) \
        .option("mergeSchema", "true") \
        .saveAsTable("dbw_banco_orion.bronze.clientes")

        #Salvar contas
        df_contas.write.format("delta").mode("append") \
        .option("path", Config.bronze_path_contas) \
        .option("mergeSchema", "true") \
        .saveAsTable("dbw_banco_orion.bronze.contas")

        #Salvar emprestimos
        df_emprestimos.write.format("delta").mode("append") \
        .option("path", Config.bronze_path_emprestimos) \
        .option("mergeSchema", "true") \
        .saveAsTable("dbw_banco_orion.bronze.emprestimos")

        #Salvar pagamentos
        df_pagamentos.write.format("delta").mode("append") \
        .option("path", Config.bronze_path_pagamentos) \
        .option("mergeSchema", "true") \
        .saveAsTable("dbw_banco_orion.bronze.pagamentos")

        #Salvar transferencias
        df_transferencias.write.format("delta").mode("append") \
        .option("path", Config.bronze_path_transferencias) \
        .option("mergeSchema", "true") \
        .saveAsTable("dbw_banco_orion.bronze.transferencias")

        print("Dados salvos com sucesso!")
        return True
    except Exception as e:
        print(f"Error: {e}")
        return False

df_cliente = spark.read.parquet("abfss://raw@stbancoorion01.dfs.core.windows.net/clientes.parquet")
df_contas = spark.read.parquet("abfss://raw@stbancoorion01.dfs.core.windows.net/contas.parquet")
df_emprestimos = spark.read.parquet("abfss://raw@stbancoorion01.dfs.core.windows.net/emprestimos.parquet")
df_pagamentos = spark.read.parquet("abfss://raw@stbancoorion01.dfs.core.windows.net/pagamentos.parquet")
df_transferencias = spark.read.parquet("abfss://raw@stbancoorion01.dfs.core.windows.net/transferencias.parquet")
if not salvar_delta(df_cliente, df_contas, df_emprestimos, df_pagamentos, df_transferencias):
    raise Exception("Error ao salvar os dados na delta")
